# Analysis

## Setting parameters

In [110]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np

In [159]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass',
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     'winequality-red',
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
inclusion_test = [
        'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # cannot retrieve attribute x, has always been on this dataset
     # 'crx', # cat feats
     'dermatology',
     # 'ecoli', # error on 1 fold
     # 'german', # cat feats
     # 'glass',  # error on 1 fold
     # 'haberman', # cannot retrieve attribute x, has always been on this dataset
     'heart',
     'ionosphere',
     # 'mammographic',  # cannot retrieve attribute x, has always been on this dataset
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     # 'winequality-red',  # too slow for now (13/09)
     'wisconsin',
     # 'yeast'  # errors on some folds
]

In [144]:
# data_sets = inclusion_test

In [160]:
len(data_sets)

18

In [7]:
data_base = Path.cwd() / 'keel-data'
include_sets = data_sets
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd() / 'results'

## Loading MODLEM results

In [8]:
weka_folder = Path.cwd() / 'weka_rules_files'
name = 'modlem'
file = 'weka-balanced-accuracy.csv'  # 'weka-accuracy.csv'

In [9]:
scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]

In [10]:
rules[name] = {}
attributes[name] = {}
for file in weka_folder.iterdir():
    if file.name[-3:] == 'txt':
        short_name = file.name[:-4]
        with open(file, 'r') as f:
            line = f.readline()
            nrs = []
            while len(line) > 4:
                nrs.append(line.count('&') + 1)
                line = f.readline()
        rules[name][short_name] = len(nrs)
        attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [13]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [14]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=include_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=include_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=include_sets
    )

## Adding results for non rule based models


In [15]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [16]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=include_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [17]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in include_sets}

## NORI Models

In [161]:
nori_models = {
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    'lower-new-check' : Path.cwd() / 'lower-approx-new-impl-test',
    # 'lower-incl-t1': Path.cwd() / 'lower-approx-luka-impl-incl.99'
}

In [162]:
for model, path in nori_models.items():
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )

In [163]:
scores.keys()

dict_keys(['modlem', 'qr', 'lin-owa', 'trun-lin-owa', 'trun-exp-owa', 'avg-qr', 'avg-lin-owa', 'prune-qr', 'prune-lin-owa', 'prune-trun-lin-owa', 'prune-trun-exp-owa', 'naive-bayes', 'svm', 'tree', 'frnn', 'lower-new-check', 'lower-incl-t1'])

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [164]:
 names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    6: [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        # 'lower-incl-t1',
        'modlem',
        'qr',
    ]
}

In [148]:
nr = 6

In [149]:
main_folder = 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [150]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [165]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]

In [166]:
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)

In [167]:
table_acc

,lower-new-check,lower-incl-t1,modlem,qr
australian,0.787517,0.813668,0.861,0.865348
bands,0.675141,0.614002,0.6035,0.533045
bupa,0.600724,0.603066,0.6525,0.5
cleveland,0.290397,0.334214,0.2764,0.208347
dermatology,0.895363,0.90482,0.941333,0.521995
heart,0.7675,0.795833,0.7605,0.785833
ionosphere,0.912468,0.909748,0.8905,0.658974
pima,0.687633,0.650047,0.6985,0.502775
sonar,0.786212,0.77197,0.7045,0.522601
spectfheart,0.628874,0.615974,0.6015,0.559156


In [28]:
bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_60233/3326315922.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [154]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)

In [155]:
table_rule

,lower-new-check,lower-incl-t1,modlem,qr
australian,92.3,57.0,121,732.2
bands,59.0,37.6,113,222.7
bupa,76.7,45.7,103,332.4
cleveland,102.2,82.6,95,331.1
dermatology,20.2,19.1,27,90.5
heart,44.7,38.4,62,303.7
ionosphere,19.3,14.8,30,460.5
pima,125.0,71.7,191,824.6
sonar,17.8,15.6,63,276.5
spectfheart,24.5,15.6,55,126.0


In [31]:
bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.1f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_60233/1993066615.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [168]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)

In [169]:
table_attribute

,lower-new-check,modlem,qr
australian,5.154734,2.355372,8.04978
bands,6.292805,2.106195,1.753697
bupa,4.202797,2.194175,4.40226
cleveland,5.449101,2.589474,8.08194
dermatology,4.695735,2.555556,6.726801
ecoli,3.854646,2.339286,3.816832
glass,3.884301,2.1,5.343654
heart,4.904402,2.209677,7.580314
ionosphere,4.527456,1.533333,11.653025
pima,5.080997,2.073298,5.687857


In [120]:
bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min'), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_60233/3339941083.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [57]:
from scipy import stats
import scikit_posthocs as sph

In [105]:
subject = table_rule

In [106]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')

In [107]:
no_mean

,lower-new-check,modlem,qr
australian,92.3,121,732.2
bands,59.0,113,222.7
bupa,76.7,103,332.4
cleveland,102.2,95,331.1
dermatology,20.2,27,90.5
ecoli,56.8,56,315.0
glass,51.1,50,225.7
heart,44.7,62,303.7
ionosphere,19.3,30,460.5
pima,125.0,191,824.6


In [104]:
stats.wilcoxon(no_mean['lower-new-check'], no_mean['modlem'], alternative="greater")


WilcoxonResult(statistic=104.0, pvalue=0.22114944458007812)

In [108]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=45.31578947368419, pvalue=0.000218491354338183)

In [109]:
post_hocs = sph.posthoc_nemenyi_friedman(no_mean.values) # , p_adjust='Holm'
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,lower-new-check,modlem,qr
lower-new-check,1.000000,0.377902,0.001
modlem,0.377902,1.000000,0.001
qr,0.001000,0.001000,1.000
